# 🔄 Step 2: Preprocessing & NDVI Calculation

Calculate NDVI and perform automated pixel classification.

## What This Notebook Does:
- ✅ Load bands from previous notebook
- ✅ Calculate NDVI (Normalized Difference Vegetation Index)
- ✅ Visualize NDVI distribution
- ✅ Apply thresholding for automated classification
- ✅ Export training data to CSV

---

**Previous:** [01_data_loading.ipynb](01_data_loading.ipynb)  
**Next:** [03_model_training.ipynb](03_model_training.ipynb)

## Load Libraries and Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle

# Load bands from previous notebook
print("📂 Loading bands from previous notebook...")
with open('outputs/loaded_bands.pkl', 'rb') as f:
    bands_data = pickle.load(f)

B02 = bands_data['B02']
B03 = bands_data['B03']
B04 = bands_data['B04']
B08 = bands_data['B08']

print(f"✅ Bands loaded! Shape: {bands_data['shape']}")

## Calculate NDVI

**NDVI Formula:** `(NIR - Red) / (NIR + Red)`

**Interpretation:**
- NDVI > 0.3: Healthy vegetation (Seagrass)
- 0 ≤ NDVI ≤ 0.3: Bare soil / Sand
- NDVI < 0: Water bodies

In [ ]:
# Calculate NDVI
ndvi = (B08 - B04) / (B08 + B04 + 1e-10)  # Add epsilon to avoid division by zero

print(f"NDVI Statistics:")
print(f"  Min:  {ndvi.min():.4f}")
print(f"  Max:  {ndvi.max():.4f}")
print(f"  Mean: {ndvi.mean():.4f}")
print(f"  Std:  {ndvi.std():.4f}")

# Visualize NDVI
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
im = plt.imshow(ndvi, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im, label='NDVI Value')
plt.title('NDVI Map', fontsize=14, fontweight='bold')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.hist(ndvi.flatten(), bins=100, edgecolor='black', alpha=0.7, color='green')
plt.xlabel('NDVI Value')
plt.ylabel('Frequency')
plt.title('NDVI Distribution', fontsize=14, fontweight='bold')
plt.axvline(x=0, color='blue', linestyle='--', linewidth=2, label='Water threshold (0)')
plt.axvline(x=0.3, color='darkgreen', linestyle='--', linewidth=2, label='Vegetation threshold (0.3)')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/02_ndvi_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n💾 NDVI analysis saved to 'outputs/02_ndvi_analysis.png'")

## Automated Pixel Classification

In [ ]:
# Apply thresholding for classification
labels = np.zeros(ndvi.shape)
labels[ndvi > 0.3] = 1    # Seagrass
labels[ndvi < 0] = 2      # Water
labels[(ndvi >= 0) & (ndvi <= 0.3)] = 3  # Sand

class_names = {0: 'Cloud/No Data', 1: 'Seagrass', 2: 'Water', 3: 'Sand'}

# Count pixels
unique, counts = np.unique(labels, return_counts=True)
print("Classification Results:")
for label, count in zip(unique, counts):
    percentage = (count / labels.size) * 100
    print(f"  {class_names[label]:15s}: {count:10,} pixels ({percentage:5.2f}%)")

# Visualize
from matplotlib.colors import ListedColormap
colors = ['gray', 'green', 'blue', 'yellow']
cmap = ListedColormap(colors)

plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)
im = plt.imshow(labels, cmap=cmap, vmin=0, vmax=3)
plt.title('Classified Coastal Areas', fontsize=14, fontweight='bold')
plt.axis('off')
cbar = plt.colorbar(im, ticks=[0, 1, 2, 3])
cbar.set_ticklabels(['Cloud', 'Seagrass', 'Water', 'Sand'])

plt.subplot(1, 2, 2)
plt.bar([class_names[l] for l in unique], counts, 
        color=[colors[int(l)] for l in unique], edgecolor='black', linewidth=2)
plt.ylabel('Pixel Count')
plt.title('Class Distribution', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/02_classification.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n💾 Classification saved to 'outputs/02_classification.png'")

## Export Training Data

In [ ]:
# Flatten and create DataFrame
data = np.stack([
    B02.flatten(), 
    B03.flatten(), 
    B04.flatten(), 
    B08.flatten(), 
    ndvi.flatten()
], axis=1)

labels_flat = labels.flatten()

df = pd.DataFrame(data, columns=["B02", "B03", "B04", "B08", "NDVI"])
df["Label"] = labels_flat.astype(int)

print("Training Data Preview:")
print(df.head(10))
print(f"\nDataset shape: {df.shape}")

# Export to CSV
df.to_csv("outputs/training_data.csv", index=False)
print("\n✅ Training data exported to 'outputs/training_data.csv'")

# Also save NDVI and labels for visualization
preprocessing_data = {
    'ndvi': ndvi,
    'labels': labels,
    'shape': ndvi.shape,
    'class_names': class_names
}

with open('outputs/preprocessing_data.pkl', 'wb') as f:
    pickle.dump(preprocessing_data, f)

print("✅ Preprocessing data saved to 'outputs/preprocessing_data.pkl'")
print("\n" + "="*60)
print("✅ PREPROCESSING COMPLETE!")
print("="*60)
print("\n📌 Next Step: Open 03_model_training.ipynb to train the CNN")